# Elementos visuais da monografia (estágio 17)

Produz as figuras e as tabelas que ilustram os capítulos de metodologia e de resultados da monografia. Cada figura comunica uma única mensagem, em português, com valores anotados e unidades de leitura imediata (hectares, porcentagem, número de dobras); as tabelas reescrevem os números da síntese técnica do estágio 15 com cabeçalhos em português e uma coluna de interpretação em linguagem comum.

O estágio não retreina, não avalia e não recalcula métricas: lê os artefatos vigentes dos estágios 03–15 e reutiliza as seleções determinísticas dos estágios 12–14, de modo que o mesmo talhão aparece nas figuras qualitativas, nos mapas de erro e nos mapas de explicabilidade. As figuras são gravadas em PDF vetorial e PNG de 300 dpi, acompanhadas das legendas prontas (`captions.tex` e `captions.md`).

## Bootstrap do workspace

O primeiro passo baixa e executa `src/bootstrap.py` (somente stdlib) — necessário porque o `src/` ainda não está disponível para import em uma sessão nova. O bootstrap obtém o repositório público, extrai `src/`, `data/external/` e `requirements-runtime.txt` para o workspace e adiciona o workspace ao `sys.path`. O `reload` garante que reexecuções usem a versão mais recente baixada.

In [ ]:
# Baixa e executa o bootstrap do workspace (etapa prévia ao import de src/).
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/jotap1101/tcc-lab/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

# Recarrega o módulo para não reutilizar uma versão antiga em cache no kernel.
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)

workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")

## Dependências pinadas

Garante as versões fixadas em `requirements-runtime.txt`, o mesmo conjunto nas duas plataformas. Quando uma biblioteca com extensão compilada já importada (numpy, scipy, torch, entre outras) diverge da versão pinada, o kernel é reiniciado automaticamente para evitar inconsistência de ABI; nesse caso, o notebook é reexecutado a partir da primeira célula.

In [ ]:
# Garante as versões pinadas e reinicia o kernel quando a ABI carregada divergir.
from src.setup import install_runtime_requirements

requirements = pathlib.Path(workspace) / "requirements-runtime.txt"
install_runtime_requirements(requirements)

## Pacote compartilhado, plataforma e armazenamento

Importa o pacote `src/` (já entregue pelo bootstrap), identifica a plataforma pela abstração em `src/io.py`, garante a raiz `MyDrive/tcc/` e resolve os caminhos de armazenamento definidos em `src/config.yaml`. O diretório das figuras e tabelas da monografia é derivado de `artifacts/`, e a fonte ilustrada é a mesma escolhida manualmente na fase de ground truth.

In [ ]:
# Resolve o contexto de runtime e o diretório canônico da monografia.
from src.data.source_decision import chosen_source
from src.monograph import inventory_path, monograph_dir
from src.setup import initialize

ctx = initialize(workspace)
platform = ctx.platform
storage_paths = ctx.storage_paths
config = ctx.config
source_name = chosen_source()
print(f"Plataforma: {platform}")
print(f"Fonte ilustrada (decisão manual): {source_name}")
print(f"Diretório da monografia: {monograph_dir(storage_paths, source_name)}")
print(f"Inventário: {inventory_path(storage_paths, source_name)}")

## Reprodutibilidade

Fixa as sementes de python/numpy/torch/cuda e habilita as flags determinísticas do PyTorch, além de registrar as versões do ambiente. O estágio apenas lê artefatos e renderiza figuras em CPU; não há treinamento nem inferência de modelos aqui.

In [ ]:
# Fixa sementes, flags determinísticas e registra o ambiente.
from src.utils import log_environment, setup_reproducibility

setup_reproducibility(config)
log_environment()

## Dependências dos estágios anteriores

Verifica o manifesto da fonte escolhida, os resumos vigentes do estágio 11 para U-Net e SegFormer, a comparação do estágio 12 e as execuções de treino dos estágios 09/10. Os mapas de explicabilidade dos estágios 13/14 são opcionais: sem eles, a figura e a tabela de XAI são omitidas das legendas.

In [ ]:
# Verifica as dependências obrigatórias antes da renderização.
from src import comparative
from src.data.patch_generation import manifest_path
from src.evaluation import evaluation_summary_path
from src.utils import check_dependencies

check_dependencies(
    {
        "Manifesto (estágio 06)": manifest_path(storage_paths, source_name),
        "Resumo da U-Net (estágio 11)": evaluation_summary_path(
            storage_paths, "unet", source_name
        ),
        "Resumo do SegFormer (estágio 11)": evaluation_summary_path(
            storage_paths, "segformer", source_name
        ),
        "Comparação (estágio 12)": comparative.comparison_path(storage_paths, source_name),
    }
)

## Geração das figuras, tabelas e legendas

Gera as figuras didáticas dos capítulos de metodologia e de resultados, as tabelas com leitura em linguagem comum e os arquivos de legendas (`captions.tex` com ambientes `figure`/`table` e `captions.md`). Tudo é persistido em `artifacts/monograph/<fonte>/` e reutilizado apenas enquanto o fingerprint das entradas (manifesto, split, avaliações, comparação e mapas de XAI) coincide.

In [ ]:
# Gera (ou reutiliza) figuras, tabelas e legendas da monografia.
from src.monograph import run_monograph

monograph_result = run_monograph(storage_paths, source_name=source_name)
print(f"Artefatos reutilizados: {monograph_result.skipped}")
print(f"Figuras: {len(monograph_result.figures)} | Tabelas: {len(monograph_result.tables)}")

## Figuras de metodologia

Exibe as figuras que explicam onde o estudo foi realizado, como o dado chega ao modelo, como o rótulo é construído, como a validação espacial é organizada, como as duas arquiteturas diferem e como a função de perda é composta.

In [ ]:
# Exibe as figuras do capítulo de metodologia.
from IPython.display import Image, Markdown, display

from src.monograph import FIGURE_CATALOG

for key, files in monograph_result.figures.items():
    if FIGURE_CATALOG[key].section != "Metodologia":
        continue
    display(Markdown(f"**{key}**"))
    display(Image(filename=str(files["png"])))

## Figuras de resultados

Exibe as figuras que respondem às perguntas do capítulo de resultados: como os modelos se saem no conjunto de validação, como os erros se distribuem entre talhões e dobras, quanta área de café cada modelo mapeia e para onde os modelos olham nos mapas de explicabilidade.

In [ ]:
# Exibe as figuras do capítulo de resultados.

from IPython.display import Image, Markdown, display

for key, files in monograph_result.figures.items():
    if FIGURE_CATALOG[key].section != "Resultados":
        continue
    display(Markdown(f"**{key}**"))
    display(Image(filename=str(files["png"])))

## Tabelas prontas para a monografia

Exibe as tabelas em formato de leitura: composição do dataset, protocolo de treinamento, glossário das métricas, desempenho dos modelos, comparação pareada com interpretação, área em hectares e relevância dos mapas de explicabilidade. As versões CSV (separador `;`) e LaTeX ficam em `tables/`.

In [ ]:
# Exibe as tabelas didáticas geradas para a monografia.
for key, table in monograph_result.tables_data.items():
    display(Markdown(f"**{key}**"))
    display(table)

## Legendas e resumo da etapa

Exibe as legendas em Markdown (as mesmas gravadas em LaTeX) e os números-chave do estágio, além do caminho de cada artefato. As legendas já trazem numeração, capítulo e rótulos (`fig:<chave>` e `tab:<chave>`) para uso direto no documento.

In [ ]:
# Exibe as legendas e o resumo da etapa.

from IPython.display import Markdown, display

from src.utils import print_summary

captions_markdown = monograph_result.captions_md_path.read_text(encoding="utf-8")
display(Markdown(captions_markdown))

summary = {
    "Fonte": monograph_result.source_name,
    "Ano de referência": monograph_result.summary.get("year", ""),
    "Patches de validação": monograph_result.summary["n_patches"],
    "IoU U-Net (micro)": round(float(monograph_result.summary["unet_iou"]), 4),
    "IoU SegFormer (micro)": round(float(monograph_result.summary["segformer_iou"]), 4),
    "Delta de IoU (SegFormer - U-Net)": round(float(monograph_result.summary["iou_delta"]), 4),
    "Figuras": len(monograph_result.figures),
    "Tabelas": len(monograph_result.tables),
    "Legendas (LaTeX)": str(monograph_result.captions_tex_path),
    "Inventário": str(monograph_result.inventory_path),
}
print_summary(summary, "17")